[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/AI-in-healthcare-book/blob/main/notebooks/chapter_08/notebook_8_6_summarization.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 8.6: Clinical Text Summarization

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand extractive vs. abstractive summarization approaches
2. Implement TF-IDF and TextRank for extractive summarization
3. Use LLMs for abstractive clinical summarization
4. Evaluate summaries with ROUGE metrics and clinical quality criteria
5. Handle different clinical document types (progress notes, discharge summaries)
6. Implement domain-specific summarization constraints

## Clinical Context

**The Challenge**: Clinicians face information overload:
- Average ICU patient generates 236 pages of documentation per day
- Physicians spend 49% of time on documentation
- Critical information buried in lengthy notes
- Handoff errors due to incomplete information transfer

**The Solution**: Automated clinical text summarization can:
- Generate concise patient summaries for handoffs
- Extract key findings from radiology reports
- Create discharge summaries from progress notes
- Identify actionable items from clinical narratives

**Real-World Impact**:
- Mayo Clinic uses AI summarization to reduce documentation time by 30%
- Stanford Hospital implements summarization for discharge summaries (95% clinician satisfaction)
- Research shows 40% reduction in handoff errors with structured summaries

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# For extractive summarization
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")
print("\nNote: This notebook demonstrates summarization with both extractive and abstractive methods.")

## Part 1: Understanding Clinical Summarization

### Extractive vs. Abstractive

**Extractive Summarization**:
- Selects important sentences from original text
- No new text generated
- Pros: Factually accurate, grammatically correct
- Cons: Can be redundant, less fluent
- Methods: TF-IDF, TextRank, BERT-based

**Abstractive Summarization**:
- Generates new text paraphrasing original content
- Uses language models to rephrase
- Pros: More fluent, concise, human-like
- Cons: Risk of hallucination, requires validation
- Methods: Seq2Seq, Transformer models, LLMs

### Clinical Document Types

1. **Progress Notes**: Daily patient updates (summarize to 3-5 key points)
2. **Discharge Summaries**: Comprehensive stay summary (extract diagnosis, procedures, follow-up)
3. **Radiology Reports**: Imaging findings (highlight critical findings)
4. **Pathology Reports**: Lab results (extract abnormal values)
5. **Consultation Notes**: Specialist recommendations (identify action items)

In [ ]:
@dataclass
class ClinicalDocument:
    """Represents a clinical document to be summarized."""
    doc_id: str
    doc_type: str
    content: str
    metadata: Dict
    reference_summary: Optional[str] = None  # Gold standard summary for evaluation

@dataclass
class Summary:
    """Represents a generated summary."""
    doc_id: str
    method: str
    summary_text: str
    sentences: List[str]
    compression_ratio: float
    metadata: Dict

print("Summarization data structures defined!")

## Part 2: Synthetic Clinical Documents

For demonstration, we'll create realistic clinical documents.

In [ ]:
# Synthetic clinical documents
clinical_documents = [
    ClinicalDocument(
        doc_id="PROG_001",
        doc_type="progress_note",
        content="""
Patient: 67-year-old male, admitted 3 days ago with acute exacerbation of COPD.

Subjective: Patient reports improved dyspnea compared to yesterday. He is able to speak in full sentences without difficulty.
Denies chest pain or palpitations. Some persistent cough with minimal clear sputum. Patient is eager to go home.
Sleep was fair, interrupted twice for vitals. Appetite improving, ate 75% of breakfast. No nausea or vomiting.

Objective: Vitals stable. Temperature 37.2°C, BP 138/82, HR 88, RR 18, SpO2 94% on 2L NC (goal >92%).
Physical exam: Alert and oriented x3. Lungs: decreased breath sounds bilaterally, no wheezes today (significant improvement).
Cardiac: regular rate and rhythm, no murmurs. Abdomen soft, non-tender. No peripheral edema. No acute distress.

Labs: WBC 8.2 (down from 12.4), Cr 1.1 (stable), BNP 180. CXR showed interval improvement, no new infiltrates.

Assessment/Plan:
1. COPD exacerbation - improving. Continue albuterol/ipratropium nebs q4h. Prednisone day 3 of 5. Consider discharge tomorrow if remains stable.
2. Mild hypertension - continue home lisinopril 10mg daily
3. Monitor for signs of pneumonia given smoking history
4. Physical therapy consult for pulmonary rehab referral before discharge
5. Smoking cessation counseling provided, patient receptive
6. Discharge planning: arrange home O2, follow-up with pulmonology in 1 week
        """,
        metadata={"date": "2024-01-15", "provider": "Dr. Smith"},
        reference_summary="67M with COPD exacerbation showing improvement. Plan: discharge tomorrow if stable, home O2, pulmonology follow-up."
    ),
    ClinicalDocument(
        doc_id="DISCHARGE_001",
        doc_type="discharge_summary",
        content="""
Patient: 82-year-old female
Admission Date: January 10, 2024
Discharge Date: January 18, 2024
Length of Stay: 8 days

Chief Complaint: Hip fracture following mechanical fall

History of Present Illness:
Patient is an 82-year-old woman with history of osteoporosis who presented to ED after falling at home while reaching for object on high shelf.
She landed on her right side and was unable to ambulate due to severe right hip pain. No loss of consciousness. No head trauma.
EMS transported her to our facility. In ED, initial assessment revealed shortened and externally rotated right lower extremity.
Plain films confirmed right femoral neck fracture. Patient was admitted for surgical management.

Past Medical History:
- Osteoporosis (on alendronate)
- Hypertension (controlled)
- Type 2 diabetes mellitus (HbA1c 6.8%)
- Hypothyroidism
- Chronic kidney disease stage 2

Hospital Course:
Patient underwent right hip hemiarthroplasty on hospital day 2 without complications. Procedure performed by orthopedic surgery.
Post-operative course unremarkable. Pain initially managed with oxycodone, transitioned to acetaminophen by day 5.
Physical therapy initiated on post-op day 1. Patient progressed from walker to cane with good strength and balance.
She achieved all mobility milestones including stairs. No signs of surgical site infection.
DVT prophylaxis with enoxaparin throughout admission. Home medications continued with dose adjustments for renal function.

Discharge Condition:
Stable, ambulating with cane, pain controlled on oral analgesics.

Discharge Medications:
1. Acetaminophen 650mg PO Q6H PRN pain
2. Alendronate 70mg PO weekly (resume)
3. Lisinopril 10mg PO daily
4. Metformin 500mg PO BID
5. Levothyroxine 50mcg PO daily
6. Enoxaparin 40mg SC daily x 2 weeks

Follow-up:
- Orthopedic surgery clinic in 2 weeks for wound check and X-ray
- Primary care in 4 weeks
- Home health PT 3x per week
- Remove surgical staples in 14 days

Patient Education:
Discussed hip precautions (no bending >90 degrees, no crossing legs, no internal rotation). Fall prevention strategies reviewed.
Importance of calcium and vitamin D supplementation emphasized. Signs of DVT/PE discussed. Patient and family demonstrated understanding.
        """,
        metadata={"date": "2024-01-18", "provider": "Dr. Johnson"},
        reference_summary="82F admitted for R femoral neck fracture s/p mechanical fall. Underwent successful R hip hemiarthroplasty. D/C home with PT, f/u ortho in 2 weeks."
    ),
    ClinicalDocument(
        doc_id="RAD_001",
        doc_type="radiology_report",
        content="""
Exam: CT CHEST WITH CONTRAST
Date: January 15, 2024
Indication: 58-year-old male with persistent cough, weight loss, and hemoptysis. Rule out malignancy.

Technique: Helical CT of chest performed before and after IV contrast administration. 120mL Omnipaque 350 used.
Patient tolerated procedure well, no adverse reaction.

Comparison: Prior chest CT from 6 months ago available for comparison.

Findings:

LUNGS: There is a 3.2 x 2.8 cm spiculated mass in the right upper lobe, increased from 1.8 cm on prior study (concerning for malignancy).
The mass abuts the pleura but does not demonstrate definite pleural invasion. No endobronchial lesion.
Multiple small pulmonary nodules in both lungs, largest 8mm in left lower lobe (concerning for metastases).
No pneumothorax. No significant pleural effusion. Mild emphysematous changes, worse than prior.

MEDIASTINUM: Enlarged right paratracheal lymph node measuring 2.1 cm (pathologic, increased from 1.2 cm).
Subcarinal lymph node measures 1.8 cm (borderline, stable). No pericardial effusion. Heart size normal.

CHEST WALL: No chest wall invasion. No destructive rib lesions.

UPPER ABDOMEN: Limited views show normal liver, spleen, and adrenal glands. Left kidney demonstrates 1.2 cm simple cyst, unchanged.
No suspicious liver lesions.

BONES: No acute fracture or suspicious lytic/blastic lesions in visualized thoracic spine.

Impression:
1. 3.2 cm spiculated right upper lobe mass with interval growth - highly suspicious for primary lung malignancy (likely non-small cell lung cancer)
2. Multiple bilateral pulmonary nodules - concerning for metastatic disease
3. Enlarged right paratracheal lymph node - suspicious for nodal metastases
4. Recommend PET/CT for staging and tissue biopsy (CT-guided or bronchoscopy with EBUS)
5. Urgent pulmonology and oncology referrals recommended

CRITICAL FINDINGS called to referring physician Dr. Williams at 2:15 PM on 1/15/2024.
        """,
        metadata={"date": "2024-01-15", "radiologist": "Dr. Chen"},
        reference_summary="58M with RUL spiculated mass (3.2cm, growing), bilateral nodules, enlarged lymph nodes. Highly concerning for lung cancer with mets. Urgent biopsy and oncology referral needed."
    )
]

print(f"Created {len(clinical_documents)} synthetic clinical documents")
print("\nDocument types:")
for doc in clinical_documents:
    print(f"  - {doc.doc_id} ({doc.doc_type}): {len(doc.content)} characters")

## Part 3: Extractive Summarization

### Method 1: TF-IDF Sentence Ranking

In [ ]:
class TFIDFSummarizer:
    """
    Extractive summarization using TF-IDF to rank sentences.
    Selects sentences with highest TF-IDF scores.
    """

    def __init__(self, num_sentences: int = 3):
        self.num_sentences = num_sentences

    def _preprocess_text(self, text: str) -> List[str]:
        """Split text into sentences."""
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()

        # Split into sentences (simple approach)
        sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)

        # Filter out very short sentences
        sentences = [s.strip() for s in sentences if len(s.split()) > 5]

        return sentences

    def summarize(self, document: ClinicalDocument) -> Summary:
        """Generate extractive summary using TF-IDF."""
        # Split into sentences
        sentences = self._preprocess_text(document.content)

        if len(sentences) <= self.num_sentences:
            # Document already short enough
            summary_sentences = sentences
        else:
            # Compute TF-IDF
            vectorizer = TfidfVectorizer(stop_words='english')
            tfidf_matrix = vectorizer.fit_transform(sentences)

            # Calculate sentence scores (sum of TF-IDF values)
            sentence_scores = tfidf_matrix.sum(axis=1).A1

            # Get top N sentences
            top_indices = sentence_scores.argsort()[-self.num_sentences:][::-1]

            # Sort by original order to maintain flow
            top_indices_sorted = sorted(top_indices)
            summary_sentences = [sentences[i] for i in top_indices_sorted]

        # Create summary
        summary_text = " ".join(summary_sentences)
        compression_ratio = len(summary_text) / len(document.content)

        return Summary(
            doc_id=document.doc_id,
            method="TF-IDF",
            summary_text=summary_text,
            sentences=summary_sentences,
            compression_ratio=compression_ratio,
            metadata={"num_original_sentences": len(sentences)}
        )

# Test TF-IDF summarizer
tfidf_summarizer = TFIDFSummarizer(num_sentences=3)

print("TF-IDF Extractive Summaries:\n")
for doc in clinical_documents:
    summary = tfidf_summarizer.summarize(doc)
    print(f"Document: {doc.doc_id} ({doc.doc_type})")
    print(f"Compression: {summary.compression_ratio:.2%}")
    print(f"Summary: {summary.summary_text[:200]}...")
    print("-" * 80)
    print()

### Method 2: TextRank (Graph-Based)

In [ ]:
class TextRankSummarizer:
    """
    Extractive summarization using TextRank algorithm.
    Creates graph of sentences, ranks by PageRank.
    """

    def __init__(self, num_sentences: int = 3, similarity_threshold: float = 0.1):
        self.num_sentences = num_sentences
        self.similarity_threshold = similarity_threshold

    def _preprocess_text(self, text: str) -> List[str]:
        """Split text into sentences."""
        text = re.sub(r'\s+', ' ', text).strip()
        sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text)
        sentences = [s.strip() for s in sentences if len(s.split()) > 5]
        return sentences

    def _build_similarity_matrix(self, sentences: List[str]) -> np.ndarray:
        """Build sentence similarity matrix using TF-IDF + cosine similarity."""
        vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_matrix = vectorizer.fit_transform(sentences)
        similarity_matrix = cosine_similarity(tfidf_matrix)

        # Zero out low similarities
        similarity_matrix[similarity_matrix < self.similarity_threshold] = 0

        return similarity_matrix

    def summarize(self, document: ClinicalDocument) -> Summary:
        """Generate extractive summary using TextRank."""
        # Split into sentences
        sentences = self._preprocess_text(document.content)

        if len(sentences) <= self.num_sentences:
            summary_sentences = sentences
        else:
            # Build similarity matrix
            similarity_matrix = self._build_similarity_matrix(sentences)

            # Create graph
            nx_graph = nx.from_numpy_array(similarity_matrix)

            # Calculate PageRank scores
            scores = nx.pagerank(nx_graph)

            # Get top N sentences
            ranked_sentences = sorted(((scores[i], i) for i in range(len(sentences))), reverse=True)
            top_indices = [idx for _, idx in ranked_sentences[:self.num_sentences]]

            # Sort by original order
            top_indices_sorted = sorted(top_indices)
            summary_sentences = [sentences[i] for i in top_indices_sorted]

        # Create summary
        summary_text = " ".join(summary_sentences)
        compression_ratio = len(summary_text) / len(document.content)

        return Summary(
            doc_id=document.doc_id,
            method="TextRank",
            summary_text=summary_text,
            sentences=summary_sentences,
            compression_ratio=compression_ratio,
            metadata={"num_original_sentences": len(sentences)}
        )

# Test TextRank summarizer
textrank_summarizer = TextRankSummarizer(num_sentences=3)

print("TextRank Extractive Summaries:\n")
for doc in clinical_documents:
    summary = textrank_summarizer.summarize(doc)
    print(f"Document: {doc.doc_id} ({doc.doc_type})")
    print(f"Compression: {summary.compression_ratio:.2%}")
    print(f"Summary: {summary.summary_text[:200]}...")
    print("-" * 80)
    print()

## Part 4: Abstractive Summarization with LLMs

### Simulated Abstractive Summarization

In [ ]:
class AbstractiveSummarizer:
    """
    Simulated abstractive summarization.
    In production, this would use an LLM API.
    """

    def __init__(self, max_length: int = 150):
        self.max_length = max_length

    def _extract_key_clinical_info(self, document: ClinicalDocument) -> Dict[str, str]:
        """Extract structured clinical information."""
        content_lower = document.content.lower()

        # Extract patient info (simplified)
        age_match = re.search(r'(\d+)[-\s]year[-\s]old\s+(male|female)', content_lower)
        patient = age_match.group(0) if age_match else "patient"

        # Extract diagnoses
        diagnoses = []
        diagnosis_keywords = ['copd', 'fracture', 'diabetes', 'hypertension', 'mass', 'cancer']
        for keyword in diagnosis_keywords:
            if keyword in content_lower:
                diagnoses.append(keyword)

        # Extract procedures
        procedures = []
        procedure_keywords = ['surgery', 'hemiarthroplasty', 'biopsy', 'ct', 'discharge']
        for keyword in procedure_keywords:
            if keyword in content_lower:
                procedures.append(keyword)

        return {
            'patient': patient,
            'diagnoses': diagnoses,
            'procedures': procedures
        }

    def _construct_prompt(self, document: ClinicalDocument) -> str:
        """Construct prompt for LLM summarization."""
        doc_type_prompts = {
            'progress_note': 'Summarize this progress note in 1-2 sentences, focusing on patient status and plan.',
            'discharge_summary': 'Summarize this discharge summary in 2-3 sentences, including diagnosis, key procedures, and follow-up.',
            'radiology_report': 'Summarize this radiology report in 1-2 sentences, highlighting critical findings and recommendations.'
        }

        instruction = doc_type_prompts.get(document.doc_type, 'Summarize this clinical document concisely.')

        prompt = f"""You are a clinical documentation specialist. {instruction}

Use medical abbreviations appropriately. Be concise but include all critical information.

DOCUMENT:
{document.content}

SUMMARY:"""

        return prompt

    def summarize(self, document: ClinicalDocument) -> Summary:
        """Generate abstractive summary (simulated)."""
        # In production, call LLM API here
        prompt = self._construct_prompt(document)

        # Simulated abstractive summary using extracted info
        info = self._extract_key_clinical_info(document)

        # Generate simple template-based summary
        if document.doc_type == 'progress_note':
            summary_text = f"{info['patient'].capitalize()} with {', '.join(info['diagnoses'])} showing improvement. Plan: continue current treatment and consider discharge."
        elif document.doc_type == 'discharge_summary':
            summary_text = f"{info['patient'].capitalize()} admitted for {', '.join(info['diagnoses'])}. Underwent {', '.join(info['procedures'])}. Discharged in stable condition with follow-up arranged."
        elif document.doc_type == 'radiology_report':
            summary_text = f"{info['patient'].capitalize()} with findings concerning for {', '.join(info['diagnoses'])}. Recommend urgent biopsy and specialist referral."
        else:
            summary_text = "Clinical summary unavailable."

        compression_ratio = len(summary_text) / len(document.content)

        return Summary(
            doc_id=document.doc_id,
            method="Abstractive (Simulated)",
            summary_text=summary_text,
            sentences=[summary_text],
            compression_ratio=compression_ratio,
            metadata={"prompt": prompt[:200]}
        )

# Test abstractive summarizer
abstractive_summarizer = AbstractiveSummarizer(max_length=150)

print("Abstractive (Simulated) Summaries:\n")
for doc in clinical_documents:
    summary = abstractive_summarizer.summarize(doc)
    print(f"Document: {doc.doc_id} ({doc.doc_type})")
    print(f"Compression: {summary.compression_ratio:.2%}")
    print(f"Summary: {summary.summary_text}")
    print("-" * 80)
    print()

### Production LLM Summarization Example

In [ ]:
# Production abstractive summarization example
production_summarization_example = '''
# PRODUCTION ABSTRACTIVE SUMMARIZATION
# Using OpenRouter with Claude 3.5 Haiku

import os
import openai

# Setup OpenRouter
openai.api_base = "https://openrouter.ai/api/v1"
openai.api_key = os.getenv("OPENROUTER_API_KEY")

def summarize_clinical_document(document: ClinicalDocument, max_words: int = 50) -> str:
    """Generate abstractive summary using LLM."""

    # Construct prompt
    prompt = f"""You are a clinical documentation specialist. Summarize this {document.doc_type} in {max_words} words or less.

IMPORTANT:
- Use standard medical abbreviations
- Include: patient demographics, diagnosis, key findings, plan/recommendations
- Be concise but preserve all critical information
- Do not add information not in the original document

DOCUMENT:
{document.content}

SUMMARY:"""

    # Call LLM
    response = openai.ChatCompletion.create(
        model="anthropic/claude-3.5-haiku",
        messages=[
            {"role": "system", "content": "You are a clinical documentation specialist skilled in medical summarization."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.1,  # Low temperature for consistency
        max_tokens=200
    )

    return response.choices[0].message.content

# Usage
for doc in clinical_documents:
    summary = summarize_clinical_document(doc, max_words=50)
    print(f"{doc.doc_id}: {summary}")

# Cost estimation:
# Average clinical document: ~1000 tokens
# Summary: ~100 tokens
# Cost per summary: ~$0.00028 (with Claude 3.5 Haiku)
# 1000 summaries: ~$0.28
'''

print("Production Abstractive Summarization Example:")
print(production_summarization_example)

## Part 5: Evaluation with ROUGE Metrics

### ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

**ROUGE-1**: Overlap of unigrams (single words)
**ROUGE-2**: Overlap of bigrams (two consecutive words)
**ROUGE-L**: Longest Common Subsequence

Each ROUGE metric has:
- **Precision**: % of n-grams in summary that appear in reference
- **Recall**: % of n-grams in reference that appear in summary
- **F1**: Harmonic mean of precision and recall

In [ ]:
class ROUGEEvaluator:
    """
    Simple ROUGE metric implementation.
    For production, use `rouge-score` library.
    """

    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization."""
        return re.findall(r'\w+', text.lower())

    def _get_ngrams(self, tokens: List[str], n: int) -> Counter:
        """Extract n-grams from tokens."""
        ngrams = []
        for i in range(len(tokens) - n + 1):
            ngrams.append(tuple(tokens[i:i+n]))
        return Counter(ngrams)

    def rouge_n(self, summary: str, reference: str, n: int = 1) -> Dict[str, float]:
        """Calculate ROUGE-N scores."""
        summary_tokens = self._tokenize(summary)
        reference_tokens = self._tokenize(reference)

        summary_ngrams = self._get_ngrams(summary_tokens, n)
        reference_ngrams = self._get_ngrams(reference_tokens, n)

        # Calculate overlap
        overlap = sum((summary_ngrams & reference_ngrams).values())

        # Precision: overlap / summary n-grams
        precision = overlap / sum(summary_ngrams.values()) if summary_ngrams else 0.0

        # Recall: overlap / reference n-grams
        recall = overlap / sum(reference_ngrams.values()) if reference_ngrams else 0.0

        # F1
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        return {
            'precision': precision,
            'recall': recall,
            'f1': f1
        }

    def rouge_l(self, summary: str, reference: str) -> Dict[str, float]:
        """Calculate ROUGE-L (Longest Common Subsequence)."""
        summary_tokens = self._tokenize(summary)
        reference_tokens = self._tokenize(reference)

        m, n = len(summary_tokens), len(reference_tokens)

        # Dynamic programming for LCS
        dp = [[0] * (n + 1) for _ in range(m + 1)]

        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if summary_tokens[i-1] == reference_tokens[j-1]:
                    dp[i][j] = dp[i-1][j-1] + 1
                else:
                    dp[i][j] = max(dp[i-1][j], dp[i][j-1])

        lcs_length = dp[m][n]

        precision = lcs_length / m if m > 0 else 0.0
        recall = lcs_length / n if n > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

        return {
            'precision': precision,
            'recall': recall,
            'f1': f1
        }

    def evaluate_summary(self, summary: str, reference: str) -> Dict:
        """Calculate all ROUGE metrics."""
        return {
            'rouge-1': self.rouge_n(summary, reference, n=1),
            'rouge-2': self.rouge_n(summary, reference, n=2),
            'rouge-l': self.rouge_l(summary, reference)
        }

# Evaluate all summarization methods
rouge_evaluator = ROUGEEvaluator()

results = []
for doc in clinical_documents:
    if doc.reference_summary:
        # Evaluate TF-IDF
        tfidf_summary = tfidf_summarizer.summarize(doc)
        tfidf_scores = rouge_evaluator.evaluate_summary(
            tfidf_summary.summary_text,
            doc.reference_summary
        )

        # Evaluate TextRank
        textrank_summary = textrank_summarizer.summarize(doc)
        textrank_scores = rouge_evaluator.evaluate_summary(
            textrank_summary.summary_text,
            doc.reference_summary
        )

        # Evaluate Abstractive
        abstractive_summary = abstractive_summarizer.summarize(doc)
        abstractive_scores = rouge_evaluator.evaluate_summary(
            abstractive_summary.summary_text,
            doc.reference_summary
        )

        results.append({
            'doc_id': doc.doc_id,
            'doc_type': doc.doc_type,
            'tfidf_rouge1_f1': tfidf_scores['rouge-1']['f1'],
            'tfidf_rouge2_f1': tfidf_scores['rouge-2']['f1'],
            'textrank_rouge1_f1': textrank_scores['rouge-1']['f1'],
            'textrank_rouge2_f1': textrank_scores['rouge-2']['f1'],
            'abstractive_rouge1_f1': abstractive_scores['rouge-1']['f1'],
            'abstractive_rouge2_f1': abstractive_scores['rouge-2']['f1']
        })

# Display results
results_df = pd.DataFrame(results)
print("\nROUGE Evaluation Results:")
print(results_df.to_string(index=False))

# Calculate average scores
print("\nAverage ROUGE-1 F1 Scores:")
print(f"  TF-IDF: {results_df['tfidf_rouge1_f1'].mean():.3f}")
print(f"  TextRank: {results_df['textrank_rouge1_f1'].mean():.3f}")
print(f"  Abstractive: {results_df['abstractive_rouge1_f1'].mean():.3f}")

## Part 6: Clinical Quality Evaluation

### Beyond ROUGE: Clinical Relevance

In [ ]:
class ClinicalQualityEvaluator:
    """
    Evaluate clinical summaries for domain-specific quality.
    """

    def __init__(self):
        # Critical clinical keywords that should appear in summaries
        self.critical_keywords = {
            'progress_note': ['patient', 'plan', 'improve|stable|worsen'],
            'discharge_summary': ['discharge', 'follow-up', 'diagnosis|condition'],
            'radiology_report': ['finding', 'recommend', 'concern|suspicious']
        }

    def check_completeness(self, summary: Summary, document: ClinicalDocument) -> Dict:
        """Check if summary contains critical information."""
        summary_lower = summary.summary_text.lower()
        doc_type = document.doc_type

        # Check for critical keywords
        keyword_presence = {}
        if doc_type in self.critical_keywords:
            for keyword_pattern in self.critical_keywords[doc_type]:
                present = bool(re.search(keyword_pattern, summary_lower))
                keyword_presence[keyword_pattern] = present

        completeness_score = sum(keyword_presence.values()) / len(keyword_presence) if keyword_presence else 0.0

        return {
            'completeness_score': completeness_score,
            'keyword_presence': keyword_presence
        }

    def check_conciseness(self, summary: Summary) -> Dict:
        """Evaluate summary conciseness."""
        word_count = len(summary.summary_text.split())
        sentence_count = len(summary.sentences)

        # Ideal: 30-100 words, 2-4 sentences
        word_score = 1.0 if 30 <= word_count <= 100 else 0.5
        sentence_score = 1.0 if 2 <= sentence_count <= 4 else 0.5

        return {
            'word_count': word_count,
            'sentence_count': sentence_count,
            'word_score': word_score,
            'sentence_score': sentence_score,
            'conciseness_score': (word_score + sentence_score) / 2
        }

    def check_medical_terminology(self, summary: Summary) -> Dict:
        """Check appropriate use of medical terminology."""
        summary_lower = summary.summary_text.lower()

        # Common medical abbreviations
        abbreviations = ['mg', 'ml', 'iv', 'po', 'bid', 'tid', 'qd', 'prn', 'pt', 'htn', 'dm', 'cad']
        abbreviation_count = sum(1 for abbr in abbreviations if abbr in summary_lower)

        # Medical terms (simplified check)
        medical_terms = ['diagnosis', 'treatment', 'therapy', 'medication', 'procedure', 'symptoms', 'findings']
        medical_term_count = sum(1 for term in medical_terms if term in summary_lower)

        return {
            'abbreviation_count': abbreviation_count,
            'medical_term_count': medical_term_count,
            'terminology_score': min(1.0, (abbreviation_count + medical_term_count) / 5)
        }

    def evaluate(self, summary: Summary, document: ClinicalDocument) -> Dict:
        """Comprehensive clinical quality evaluation."""
        completeness = self.check_completeness(summary, document)
        conciseness = self.check_conciseness(summary)
        terminology = self.check_medical_terminology(summary)

        # Overall clinical quality score
        clinical_quality_score = (
            completeness['completeness_score'] * 0.4 +
            conciseness['conciseness_score'] * 0.3 +
            terminology['terminology_score'] * 0.3
        )

        return {
            'clinical_quality_score': clinical_quality_score,
            'completeness': completeness,
            'conciseness': conciseness,
            'terminology': terminology
        }

# Evaluate clinical quality
clinical_evaluator = ClinicalQualityEvaluator()

print("Clinical Quality Evaluation:\n")
for doc in clinical_documents:
    print(f"Document: {doc.doc_id} ({doc.doc_type})")
    print("-" * 80)

    # Evaluate each method
    methods = [
        ('TF-IDF', tfidf_summarizer.summarize(doc)),
        ('TextRank', textrank_summarizer.summarize(doc)),
        ('Abstractive', abstractive_summarizer.summarize(doc))
    ]

    for method_name, summary in methods:
        evaluation = clinical_evaluator.evaluate(summary, doc)
        print(f"\n{method_name}:")
        print(f"  Clinical Quality Score: {evaluation['clinical_quality_score']:.3f}")
        print(f"  - Completeness: {evaluation['completeness']['completeness_score']:.3f}")
        print(f"  - Conciseness: {evaluation['conciseness']['conciseness_score']:.3f}")
        print(f"  - Terminology: {evaluation['terminology']['terminology_score']:.3f}")
        print(f"  Word count: {evaluation['conciseness']['word_count']}")

    print("\n" + "=" * 80 + "\n")

## Part 7: Visualization

In [ ]:
# Visualize summarization performance

# Collect evaluation data
eval_data = []
for doc in clinical_documents:
    methods = [
        ('TF-IDF', tfidf_summarizer.summarize(doc)),
        ('TextRank', textrank_summarizer.summarize(doc)),
        ('Abstractive', abstractive_summarizer.summarize(doc))
    ]

    for method_name, summary in methods:
        # Clinical quality
        clin_eval = clinical_evaluator.evaluate(summary, doc)

        # ROUGE (if reference available)
        if doc.reference_summary:
            rouge_scores = rouge_evaluator.evaluate_summary(summary.summary_text, doc.reference_summary)
            rouge1_f1 = rouge_scores['rouge-1']['f1']
        else:
            rouge1_f1 = np.nan

        eval_data.append({
            'doc_id': doc.doc_id,
            'method': method_name,
            'compression_ratio': summary.compression_ratio,
            'word_count': clin_eval['conciseness']['word_count'],
            'clinical_quality': clin_eval['clinical_quality_score'],
            'rouge1_f1': rouge1_f1
        })

eval_df = pd.DataFrame(eval_data)

# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Compression ratio by method
eval_df.groupby('method')['compression_ratio'].mean().plot(kind='bar', ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('Average Compression Ratio by Method')
axes[0, 0].set_ylabel('Compression Ratio')
axes[0, 0].set_xlabel('Method')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].tick_params(axis='x', rotation=45)

# Word count by method
eval_df.groupby('method')['word_count'].mean().plot(kind='bar', ax=axes[0, 1], color='coral')
axes[0, 1].set_title('Average Word Count by Method')
axes[0, 1].set_ylabel('Word Count')
axes[0, 1].set_xlabel('Method')
axes[0, 1].axhline(50, color='red', linestyle='--', label='Target (50 words)')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].tick_params(axis='x', rotation=45)

# Clinical quality by method
eval_df.groupby('method')['clinical_quality'].mean().plot(kind='bar', ax=axes[1, 0], color='mediumseagreen')
axes[1, 0].set_title('Average Clinical Quality Score by Method')
axes[1, 0].set_ylabel('Clinical Quality Score')
axes[1, 0].set_xlabel('Method')
axes[1, 0].set_ylim(0, 1)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].tick_params(axis='x', rotation=45)

# ROUGE-1 F1 by method
rouge_data = eval_df.dropna(subset=['rouge1_f1'])
rouge_data.groupby('method')['rouge1_f1'].mean().plot(kind='bar', ax=axes[1, 1], color='orchid')
axes[1, 1].set_title('Average ROUGE-1 F1 Score by Method')
axes[1, 1].set_ylabel('ROUGE-1 F1')
axes[1, 1].set_xlabel('Method')
axes[1, 1].set_ylim(0, 1)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print("\nSummary Statistics:")
print(eval_df.groupby('method')[['compression_ratio', 'word_count', 'clinical_quality', 'rouge1_f1']].mean())

## Key Takeaways

### Summarization Approaches
1. **Extractive**: Selects important sentences from original text
   - TF-IDF: Ranks sentences by term frequency
   - TextRank: Graph-based ranking using PageRank
   - Pros: Factually accurate, preserves exact wording
   - Cons: Can be redundant, less fluent

2. **Abstractive**: Generates new text paraphrasing original
   - Uses LLMs to rephrase and condense
   - Pros: More fluent, concise, human-like
   - Cons: Risk of hallucination, requires validation

### Evaluation Metrics
1. **ROUGE Metrics** (Automatic):
   - ROUGE-1: Unigram overlap (content coverage)
   - ROUGE-2: Bigram overlap (fluency)
   - ROUGE-L: Longest common subsequence
   - Target: ROUGE-1 F1 >0.4, ROUGE-2 F1 >0.2 for clinical summaries

2. **Clinical Quality** (Domain-Specific):
   - Completeness: Contains critical clinical information
   - Conciseness: 30-100 words, 2-4 sentences
   - Medical terminology: Appropriate abbreviations and terms
   - Actionability: Includes follow-up and recommendations

### Best Practices
1. **Document-Type Specific**: Tailor summarization to document type
   - Progress notes: Focus on status and plan
   - Discharge summaries: Include diagnosis, procedures, follow-up
   - Radiology: Highlight critical findings

2. **Length Control**: Target appropriate summary length
   - Progress notes: 1-2 sentences
   - Discharge summaries: 2-3 sentences
   - Radiology reports: 1-2 sentences

3. **Clinical Validation**: Always validate summaries
   - Human review required for critical use cases
   - Check for hallucinations in abstractive summaries
   - Verify all critical information included

### Production Considerations
1. **Safety**: Use extractive for critical clinical decisions (less risk)
2. **Latency**: Extractive is faster (<100ms), abstractive slower (1-3s)
3. **Cost**: Abstractive uses LLM API (~$0.0003 per summary)
4. **Hybrid Approach**: Combine extractive (for accuracy) + abstractive (for fluency)

### Common Pitfalls
- Over-compression: Summary too short, missing critical info
- Hallucination: LLM adds information not in original
- Loss of context: Important qualifiers or negations lost
- Redundancy: Extractive summaries may repeat information

## Exercises

### Exercise 1: Implement Maximal Marginal Relevance (MMR)
Modify extractive summarizers to reduce redundancy using MMR: select sentences that are both relevant and diverse.

### Exercise 2: Section-Aware Summarization
Build a summarizer that extracts one key sentence from each section of a clinical document to ensure balanced coverage.

### Exercise 3: Aspect-Based Summarization
Create summaries focused on specific aspects: diagnosis, treatment plan, follow-up, etc.

### Exercise 4: Multi-Document Summarization
Implement summarization across multiple progress notes to create a comprehensive patient timeline.

### Exercise 5: Production LLM Summarization
Build a complete production summarizer using:
- Claude 3.5 Haiku via OpenRouter
- Custom prompts for each document type
- Hallucination detection
- Human-in-the-loop validation

### Exercise 6: Fine-tune BERT for Extractive Summarization
Fine-tune a BERT model on clinical summarization dataset (e.g., MIMIC-III notes) for sentence ranking.

### Bonus Challenge: Real-Time Summarization Dashboard
Build a web interface that:
- Accepts clinical document input
- Shows multiple summarization methods side-by-side
- Displays ROUGE and clinical quality scores
- Allows clinician feedback for continuous improvement